# S5 — WavLM-Large SER
## AudioGuard FYP | SER Track
**Purpose**: Fine-tune `microsoft/wavlm-large` for SER. WavLM is designed for diverse speech processing tasks beyond just ASR. Similar to S4 but using a more advanced base model and supporting gradient checkpointing for VRAM efficiency.

**Expected runtime**: ~30–60 min on GPU / Not recommended on CPU

**Output**: `./outputs/S5_wavlm_large_ser/`

In [ ]:
%pip install transformers datasets accelerate librosa soundfile scikit-learn pandas numpy tqdm torch torchvision torchaudio matplotlib seaborn --quiet
print("✓ Dependencies installed")

In [ ]:
import os, sys, json, time, random, glob
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import torch
import matplotlib.pyplot as plt
import seaborn as sns
 
from pathlib import Path
from tqdm.auto import tqdm
 
from transformers import (
    WavLMForSequenceClassification,
    AutoFeatureExtractor,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from datasets import Dataset, Audio, ClassLabel, Features, Value
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
)
from sklearn.model_selection import train_test_split
 
print("✓ All imports successful")

In [ ]:
CONFIG = {
    "model_id": "S5",
    "model_name": "microsoft/wavlm-large",
    "track": "SER",
    "num_labels": 7,
    "label2id": {
        "neutral":   0,
        "happy":     1,
        "sad":       2,
        "angry":     3,
        "fear":      4,
        "disgust":   5,
        "surprise":  6,
    },
    "epochs": 5,
    "batch_size": 4,                   # keep low for WavLM-Large on 16 GB VRAM
    "gradient_accumulation_steps": 4,  # effective batch = 16
    "learning_rate": 2e-5,
    "warmup_ratio": 0.1,
    "max_audio_len": 16000 * 6,        # 6 s at 16 kHz — clips longer files
    "sampling_rate": 16_000,
    "seed": 42,
    "output_dir": "/kaggle/working/",
    "logging_steps": 20,
    "eval_steps": 100,
    "save_steps": 100,
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_f1",
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}
id2label = {v: k for k, v in CONFIG["label2id"].items()}
CONFIG["id2label"] = id2label
 
os.makedirs(CONFIG["output_dir"], exist_ok=True)
print(f"Device : {CONFIG['device']}")
print(f"Output : {CONFIG['output_dir']}")
print("CONFIG loaded ✓")


In [ ]:
# ── RAVDESS ──────────────────────────────────────────────────────────────────
# Filename format: 03-01-{emotion}-{intensity}-{statement}-{repetition}-{actor}.wav
# Emotion codes: 01=neutral,02=calm,03=happy,04=sad,05=angry,06=fear,07=disgust,08=surprise
 
RAVDESS_EMOTION_MAP = {
    "01": "neutral",
    "02": "neutral",   # calm → neutral
    "03": "happy",
    "04": "sad",
    "05": "angry",
    "06": "fear",
    "07": "disgust",
    "08": "surprise",
}
 
RAVDESS_ROOT = Path("/kaggle/input/datasets/uwrfkaggler/ravdess-emotional-speech-audio")
 
def load_ravdess(root: Path) -> pd.DataFrame:
    records = []
    for wav in root.rglob("*.wav"):
        parts = wav.stem.split("-")
        if len(parts) < 7:
            continue
        emo_code = parts[2]
        label = RAVDESS_EMOTION_MAP.get(emo_code)
        if label is None or label not in CONFIG["label2id"]:
            continue
        records.append({"path": str(wav), "label": label})
    df = pd.DataFrame(records)
    print(f"RAVDESS: {len(df)} files, {df['label'].nunique()} classes")
    return df

In [ ]:
# ── TESS ─────────────────────────────────────────────────────────────────────
# Folder names encode emotion: YAF_<word>_<emotion> / OAF_<word>_<emotion>
# Emotion names in folder: angry, disgust, fear, happy, neutral, ps (=surprise), sad
 
TESS_EMOTION_MAP = {
    "angry":   "angry",
    "disgust": "disgust",
    "fear":    "fear",
    "happy":   "happy",
    "neutral": "neutral",
    "ps":      "surprise",
    "sad":     "sad",
}
 
TESS_ROOT = Path("/kaggle/input/datasets/ejlok1/toronto-emotional-speech-set-tess")
 
def load_tess(root: Path) -> pd.DataFrame:
    records = []
    for wav in root.rglob("*.wav"):
        folder = wav.parent.name.lower()   # e.g. YAF_back_angry
        emo_key = folder.split("_")[-1]
        label = TESS_EMOTION_MAP.get(emo_key)
        if label is None or label not in CONFIG["label2id"]:
            continue
        records.append({"path": str(wav), "label": label})
    df = pd.DataFrame(records)
    print(f"TESS   : {len(df)} files, {df['label'].nunique()} classes")
    return df

In [ ]:
# ── Combine ───────────────────────────────────────────────────────────────────
df_ravdess = load_ravdess(RAVDESS_ROOT)
df_tess    = load_tess(TESS_ROOT)
df = pd.concat([df_ravdess, df_tess], ignore_index=True)
df["label_id"] = df["label"].map(CONFIG["label2id"])
 
print(f"\nCombined: {len(df)} total samples")
print(df["label"].value_counts())

In [ ]:
# ## 4 · Train / Validation Split
 
# %%
random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
 
train_df, val_df = train_test_split(
    df,
    test_size=0.15,
    stratify=df["label_id"],
    random_state=CONFIG["seed"],
)
print(f"Train: {len(train_df)}  |  Val: {len(val_df)}")

In [ ]:
# ## 5 · Feature Extractor & Preprocessing
 
# %%
feature_extractor = AutoFeatureExtractor.from_pretrained(
    CONFIG["model_name"],
    sampling_rate=CONFIG["sampling_rate"],
)
print("Feature extractor loaded ✓")
 
# %%
def load_and_preprocess(batch):
    """Load audio, resample to 16 kHz, clip/pad, extract features."""
    paths  = batch["path"]
    labels = batch["label_id"]
 
    input_values_list = []
    for path in paths:
        waveform, sr = librosa.load(path, sr=CONFIG["sampling_rate"], mono=True)
        # Clip to max length
        if len(waveform) > CONFIG["max_audio_len"]:
            waveform = waveform[: CONFIG["max_audio_len"]]
        input_values_list.append(waveform)
 
    processed = feature_extractor(
        input_values_list,
        sampling_rate=CONFIG["sampling_rate"],
        padding=True,
        truncation=True,
        max_length=CONFIG["max_audio_len"],
        return_tensors="np",
    )
    processed["labels"] = labels
    return processed
 
# Build HF Datasets
def df_to_hf_dataset(dataframe: pd.DataFrame) -> Dataset:
    return Dataset.from_dict(
        {"path": dataframe["path"].tolist(), "label_id": dataframe["label_id"].tolist()}
    )
 
train_raw = df_to_hf_dataset(train_df)
val_raw   = df_to_hf_dataset(val_df)
 
print("Preprocessing train split …")
train_ds = train_raw.map(
    load_and_preprocess,
    batched=True,
    batch_size=32,
    remove_columns=["path", "label_id"],
    desc="Train",
)
 
print("Preprocessing val split …")
val_ds = val_raw.map(
    load_and_preprocess,
    batched=True,
    batch_size=32,
    remove_columns=["path", "label_id"],
    desc="Val",
)
 
train_ds.set_format("torch")
val_ds.set_format("torch")
print("Datasets ready ✓")


In [ ]:
# ## 6 · Model
 
# %%
model = WavLMForSequenceClassification.from_pretrained(
    CONFIG["model_name"],
    num_labels=CONFIG["num_labels"],
    label2id=CONFIG["label2id"],
    id2label=CONFIG["id2label"],
    ignore_mismatched_sizes=True,
)
 
# Freeze CNN feature extractor — only fine-tune transformer layers + classifier
# (freeze_feature_extractor() was removed in transformers ≥ 4.38)
for param in model.wavlm.feature_extractor.parameters():
    param.requires_grad = False
 
# Gradient checkpointing: trades compute for VRAM
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
 
print(f"Model loaded on {CONFIG['device']} ✓")
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# ## 7 · Metrics
 
# %%
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1  = f1_score(labels, preds, average="weighted", zero_division=0)
    return {"accuracy": acc, "f1": f1}

In [ ]:
# ## 8 · Training

# %%
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=feature_extractor,
    padding=True,
    return_tensors="pt",
)

training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["epochs"],
    per_device_train_batch_size=CONFIG["batch_size"],
    per_device_eval_batch_size=CONFIG["batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    warmup_steps=100,
    eval_strategy="steps",
    eval_steps=CONFIG["eval_steps"],
    save_strategy="steps",
    save_steps=CONFIG["save_steps"],
    logging_steps=CONFIG["logging_steps"],
    load_best_model_at_end=CONFIG["load_best_model_at_end"],
    metric_for_best_model=CONFIG["metric_for_best_model"],
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,
    seed=CONFIG["seed"],
    report_to="none",
    push_to_hub=False,
    save_total_limit=2,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    data_collator=data_collator,          # ← was missing
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print("Starting training …")
t0 = time.time()
trainer.train()
elapsed = time.time() - t0
print(f"\n✓ Training complete in {elapsed/60:.1f} min")

In [ ]:
# ## 9 · Evaluation & Confusion Matrix
 
# %%
print("Running final evaluation on validation set …")
eval_results = trainer.evaluate()
print(json.dumps(eval_results, indent=2))
 
# Save metrics
with open(os.path.join(CONFIG["output_dir"], "eval_results.json"), "w") as f:
    json.dump(eval_results, f, indent=2)
 
# %%
# Confusion matrix
preds_out  = trainer.predict(val_ds)
pred_ids   = np.argmax(preds_out.predictions, axis=-1)
true_ids   = preds_out.label_ids
label_names = [id2label[i] for i in range(CONFIG["num_labels"])]
 
cm = confusion_matrix(true_ids, pred_ids)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=label_names,
    yticklabels=label_names,
    cmap="Blues",
    ax=ax,
)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("S5 WavLM-Large — Confusion Matrix (Val)")
plt.tight_layout()
cm_path = os.path.join(CONFIG["output_dir"], "confusion_matrix.png")
fig.savefig(cm_path, dpi=150)
plt.show()
print(f"Confusion matrix saved → {cm_path}")
 
print("\nClassification Report:")
print(classification_report(true_ids, pred_ids, target_names=label_names, zero_division=0))


In [ ]:
# ## 10 · Save Model & Config
 
# %%
model_save_path = os.path.join(CONFIG["output_dir"], "model_final")
trainer.save_model(model_save_path)
feature_extractor.save_pretrained(model_save_path)
 
with open(os.path.join(model_save_path, "config_run.json"), "w") as f:
    cfg_serialisable = {k: v for k, v in CONFIG.items() if isinstance(v, (str, int, float, bool, dict))}
    json.dump(cfg_serialisable, f, indent=2)
 
print(f"Model saved → {model_save_path}")
print("## Training Complete ✓")